In [36]:
import pandas as pd
import numpy as np

In [37]:

df = pd.DataFrame({
    "Hours": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "Passed": [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
})


print(df)

    Hours  Passed
0       1       0
1       2       0
2       3       0
3       4       0
4       5       1
5       6       0
6       7       1
7       8       1
8       9       1
9      10       1
10     11       1
11     12       1


In [38]:
client1 = df.iloc[0:4]
client2 = df.iloc[4:8]
client3 = df.iloc[8:]


In [39]:
z = (client1['Passed'].sum()+client2['Passed'].sum()+client3['Passed'].sum())/12

In [40]:
print(z)

0.5833333333333334


In [41]:
p = np.log(z/(1-z))

In [42]:
p

np.float64(0.336472236621213)

In [43]:
client1_quantiles = np.percentile(client1['Hours'], [0, 50, 100])
client2_quantiles = np.percentile(client2['Hours'], [0, 50, 100])
client3_quantiles = np.percentile(client3['Hours'], [0, 50, 100])

print("Client 1 local summary:", client1_quantiles)
print("Client 2 local summary:", client2_quantiles)
print("Client 3 local summary:", client3_quantiles)

all_summaries = np.concatenate([client1_quantiles, client2_quantiles, client3_quantiles])
global_blueprint = np.unique(all_summaries)

print("\nGlobal Bin Blueprint (Split Thresholds):", global_blueprint)

Client 1 local summary: [1.  2.5 4. ]
Client 2 local summary: [5.  6.5 8. ]
Client 3 local summary: [ 9.  10.5 12. ]

Global Bin Blueprint (Split Thresholds): [ 1.   2.5  4.   5.   6.5  8.   9.  10.5 12. ]


In [44]:
client1['Z']=z
client2['Z']=z
client3['Z']=z

In [45]:
client1['p']=p
client2['p']=p
client3['p']=p

In [46]:
client1['yp']=1/(1+np.exp(-client1['p']))
client2['yp']=1/(1+np.exp(-client2['p']))
client3['yp']=1/(1+np.exp(-client3['p']))

In [47]:
client1['gradient']=client1['yp']-client1['Passed']
client2['gradient']=client2['yp']-client2['Passed']
client3['gradient']=client3['yp']-client3['Passed']

In [48]:
client1['hessian'] = client1['yp']*(1-client1['yp'])
client2['hessian'] = client2['yp']*(1-client2['yp'])
client3['hessian'] = client3['yp']*(1-client3['yp'])

In [49]:
# 1. Apply global binning to each client using the global blueprint
client1['bin'] = pd.cut(client1['Hours'], bins=global_blueprint, include_lowest=True)
client2['bin'] = pd.cut(client2['Hours'], bins=global_blueprint, include_lowest=True)
client3['bin'] = pd.cut(client3['Hours'], bins=global_blueprint, include_lowest=True)

# 2. Group by the bins and sum up the gradients and hessians locally
hist_client1 = client1.groupby('bin', observed=False)[['gradient', 'hessian']].sum()
hist_client2 = client2.groupby('bin', observed=False)[['gradient', 'hessian']].sum()
hist_client3 = client3.groupby('bin', observed=False)[['gradient', 'hessian']].sum()

print("--- Client 1 Local Histogram ---")
print(hist_client1)

print("\n--- Client 2 Local Histogram ---")
print(hist_client2)

print("\n--- Client 3 Local Histogram ---")
print(hist_client3)

--- Client 1 Local Histogram ---
              gradient   hessian
bin                             
(0.999, 2.5]  1.166667  0.486111
(2.5, 4.0]    1.166667  0.486111
(4.0, 5.0]    0.000000  0.000000
(5.0, 6.5]    0.000000  0.000000
(6.5, 8.0]    0.000000  0.000000
(8.0, 9.0]    0.000000  0.000000
(9.0, 10.5]   0.000000  0.000000
(10.5, 12.0]  0.000000  0.000000

--- Client 2 Local Histogram ---
              gradient   hessian
bin                             
(0.999, 2.5]  0.000000  0.000000
(2.5, 4.0]    0.000000  0.000000
(4.0, 5.0]   -0.416667  0.243056
(5.0, 6.5]    0.583333  0.243056
(6.5, 8.0]   -0.833333  0.486111
(8.0, 9.0]    0.000000  0.000000
(9.0, 10.5]   0.000000  0.000000
(10.5, 12.0]  0.000000  0.000000

--- Client 3 Local Histogram ---
              gradient   hessian
bin                             
(0.999, 2.5]  0.000000  0.000000
(2.5, 4.0]    0.000000  0.000000
(4.0, 5.0]    0.000000  0.000000
(5.0, 6.5]    0.000000  0.000000
(6.5, 8.0]    0.000000  0.000000
(8.0, 9.

In [50]:
global_histogram = hist_client1 + hist_client2 + hist_client3
print("--- Global Histogram ---")
print(global_histogram)

--- Global Histogram ---
              gradient   hessian
bin                             
(0.999, 2.5]  1.166667  0.486111
(2.5, 4.0]    1.166667  0.486111
(4.0, 5.0]   -0.416667  0.243056
(5.0, 6.5]    0.583333  0.243056
(6.5, 8.0]   -0.833333  0.486111
(8.0, 9.0]   -0.416667  0.243056
(9.0, 10.5]  -0.416667  0.243056
(10.5, 12.0] -0.833333  0.486111


In [52]:
best_gain = -1
best_split = -1
lgradi = -1
rgradi = -1
lhessi = -1
rhessi = -1
total_grad = global_histogram['gradient'].sum()
total_hess = global_histogram['hessian'].sum()
gp = total_grad**2/(total_hess+1)
for i in range(0,len(global_histogram)-1):
    lgrad = global_histogram['gradient'].iloc[:i].sum()
    rgrad = total_grad-lgrad
    lhess = global_histogram['hessian'].iloc[:i].sum()
    rhess = total_hess - lhess
    
    gain = 0.5*(lgrad**2/(lhess+1)+(rgrad**2)/(rhess+1) - gp)
    if(gain>best_gain):
        best_gain = gain
        best_split = i 
        lgradi = lgrad
        rgradi = rgrad
        lhessi = lhess
        rhessi = rhess
        
print(f"\nBest Split Index: {best_split}")
print(f"Best Gain: {best_gain}")
print(f"Split Threshold / Bin: {global_histogram.index[best_split - 1]}")


Best Split Index: 4
Best Gain: 2.5423728813559334
Split Threshold / Bin: (5.0, 6.5]


In [54]:
leftweight = -lgradi/(lhessi+1)
rightweight = -rgradi/(rhessi+1)

In [ ]:
client1['Z'] += np.where(client1['Hours']<=global_histogram.index[best_split - 1].right,0.1* leftweight,0.1*rightweight)
client1['p'] = 1/(1+np.exp(-client1['Z']))
client1['gradient'] = client1['p']-client1['Passed']
client1['hessian'] = client1['p']*(1-client1['p'])

client2['Z'] += np.where(client2['Hours']<=global_histogram.index[best_split - 1].right,0.1* leftweight,0.1*rightweight)
client2['p'] = 1/(1+np.exp(-client2['Z']))
client2['gradient'] = client2['p']-client2['Passed']
client2['hessian'] = client2['p']*(1-client2['p'])

client3['Z'] += np.where(client3['Hours']<=global_histogram.index[best_split - 1].right,0.1* leftweight,0.1*rightweight)
client3['p'] = 1/(1+np.exp(-client3['Z']))
client3['gradient'] = client3['p']-client3['Passed']
client3['hessian'] = client3['p']*(1-client3['p'])


In [61]:
print(client1)

   Hours  Passed         Z         p        yp  gradient   hessian  \
0      1       0  0.278249  0.569117  0.583333  0.569117  0.245223   
1      2       0  0.278249  0.569117  0.583333  0.569117  0.245223   
2      3       0  0.278249  0.569117  0.583333  0.569117  0.245223   
3      4       0  0.278249  0.569117  0.583333  0.569117  0.245223   

            bin  
0  (0.999, 2.5]  
1  (0.999, 2.5]  
2    (2.5, 4.0]  
3    (2.5, 4.0]  


In [62]:
def train_federated_xgboost(df, n_trees, learning_rate=0.1, reg_lambda=1.0):
    df = df.copy()

    clients = [
        df.iloc[0:4].copy(),
        df.iloc[4:8].copy(),
        df.iloc[8:].copy()
    ]

    total_rows = sum(len(client) for client in clients)
    total_positive = sum(client["Passed"].sum() for client in clients)

    base_prob = np.clip(total_positive / total_rows, 1e-7, 1 - 1e-7)
    base_score = np.log(base_prob / (1 - base_prob))

    for client in clients:
        client["Z"] = base_score
        client["p"] = base_prob

    local_quantiles = [
        np.percentile(client["Hours"], [0, 50, 100])
        for client in clients
    ]

    global_bins = np.unique(np.concatenate(local_quantiles))

    if len(global_bins) < 3:
        raise ValueError("At least three bin boundaries are required.")

    trees = []

    for tree_no in range(n_trees):
        histograms = []

        for client in clients:
            client["p"] = 1 / (1 + np.exp(-client["Z"]))
            client["gradient"] = client["p"] - client["Passed"]
            client["hessian"] = client["p"] * (1 - client["p"])

            client["bin"] = pd.cut(
                client["Hours"],
                bins=global_bins,
                include_lowest=True
            )

            hist = client.groupby(
                "bin", observed=False
            )[["gradient", "hessian"]].sum()

            histograms.append(hist)

        global_histogram = (
            pd.concat(histograms)
            .groupby(level=0)
            .sum()
        )

        total_grad = global_histogram["gradient"].sum()
        total_hess = global_histogram["hessian"].sum()

        parent_score = total_grad**2 / (total_hess + reg_lambda)

        best_gain = -np.inf
        best_split = None
        best_weights = None

        for i in range(len(global_histogram) - 1):
            left_hist = global_histogram.iloc[:i + 1]
            right_hist = global_histogram.iloc[i + 1:]

            left_grad = left_hist["gradient"].sum()
            left_hess = left_hist["hessian"].sum()

            right_grad = right_hist["gradient"].sum()
            right_hess = right_hist["hessian"].sum()

            gain = 0.5 * (
                left_grad**2 / (left_hess + reg_lambda)
                + right_grad**2 / (right_hess + reg_lambda)
                - parent_score
            )

            if gain > best_gain:
                best_gain = gain
                best_split = global_bins[i + 1]

                left_weight = -left_grad / (left_hess + reg_lambda)
                right_weight = -right_grad / (right_hess + reg_lambda)

                best_weights = (left_weight, right_weight)

        if best_split is None:
            break

        left_weight, right_weight = best_weights

        for client in clients:
            client["Z"] += learning_rate * np.where(
                client["Hours"] <= best_split,
                left_weight,
                right_weight
            )

            client["p"] = 1 / (1 + np.exp(-client["Z"]))

        trees.append({
            "tree": tree_no + 1,
            "split": best_split,
            "gain": best_gain,
            "left_weight": left_weight,
            "right_weight": right_weight
        })

    return clients, trees

In [63]:
clients, trees = train_federated_xgboost(df, n_trees=10)

print(pd.DataFrame(trees))

for i, client in enumerate(clients, start=1):
    print(f"\nClient {i}")
    print(client[["Hours", "Passed", "Z", "p"]])

   tree  split      gain  left_weight  right_weight
0     1    6.5  2.542373    -1.016949      1.016949
1     2    6.5  2.253320    -0.947989      0.968242
2     3    6.5  2.003546    -0.886447      0.924312
3     4    4.0  1.795978    -1.025999      0.720990
4     5    6.5  1.660026    -0.804976      0.854545
5     6    4.0  1.493204    -0.936239      0.665837
6     7    6.5  1.386829    -0.736256      0.794843
7     8    4.0  1.251660    -0.861079      0.617816
8     9    6.5  1.167364    -0.677475      0.742875
9    10    4.0  1.057072    -0.797114      0.575490

Client 1
   Hours  Passed        Z         p
0      1       0 -0.53258  0.369915
1      2       0 -0.53258  0.369915
2      3       0 -0.53258  0.369915
3      4       0 -0.53258  0.369915

Client 2
   Hours  Passed         Z         p
4      5       1  0.087476  0.521855
5      6       0  0.087476  0.521855
6      7       1  1.124662  0.754852
7      8       1  1.124662  0.754852

Client 3
    Hours  Passed         Z      